#  CS2 Market Index & Viewership: Advanced Hypothesis Testing

In this notebook, we perform rigorous statistical hypothesis testing to determine the validity of our assumptions regarding the Counter-Strike 2 (CS2) market. We will utilize our scraped market prices, daily Twitch viewership data, and the S-Tier/Major tournament schedule.

### Data Preparation Pipeline
1. Load market prices, Twitch viewership, and tournament schedules.
2. Calculate the **Market Index** (sum of all item prices for a given day).
3. Calculate **Daily Returns** (percentage change in prices) to measure market volatility.
4. Merge all datasets into a single cohesive dataframe.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
mega_df = pd.read_csv("dsa210_mega_data.csv")
twitch_df = pd.read_csv("cs2_gunluk_izleyici.csv")
tournaments_df = pd.read_csv("tournaments.csv")

# 2. Date Formatting
mega_df['date'] = pd.to_datetime(mega_df['date'])
twitch_df['date'] = pd.to_datetime(twitch_df['tarih']).dt.tz_localize(None).dt.normalize()
tournaments_df['start_date'] = pd.to_datetime(tournaments_df['start_date'])
tournaments_df['end_date'] = pd.to_datetime(tournaments_df['end_date'])

# 3. Calculate Market Index & Daily Returns
item_columns = [col for col in mega_df.columns if col != 'date']
mega_df['Market_Index'] = mega_df[item_columns].sum(axis=1)
mega_df['Daily_Return'] = mega_df['Market_Index'].pct_change().fillna(0)

# 4. Merge Data
df = pd.merge(mega_df[['date', 'Market_Index', 'Daily_Return']], twitch_df[['date', 'ort_izleyici']], on='date', how='left')
df = df.dropna(subset=['Market_Index', 'ort_izleyici'])

# 5. Label Tournament Periods dynamically
def get_tourney_type(d):
    for _, row in tournaments_df.iterrows():
        if row['start_date'] <= d <= row['end_date']:
            return row['tournament_type']
    return 'Normal Period'

df['Tourney_Type'] = df['date'].apply(get_tourney_type)

print("Data successfully prepared for Hypothesis Testing.")
print(df.head())

Data successfully prepared for Hypothesis Testing.
        date  Market_Index  Daily_Return  ort_izleyici   Tourney_Type
0 2023-01-01      3472.629      0.000000       55907.0  Normal Period
1 2023-01-02      4399.124      0.266799       79025.0  Normal Period
2 2023-01-03      4668.690      0.061277       70368.0  Normal Period
3 2023-01-04      5350.422      0.146022       94890.0  Normal Period
4 2023-01-05      5341.092     -0.001744       96017.0  Normal Period


## Test 1: Pearson Correlation Coefficient
**Objective:** To determine if there is a statistically significant linear relationship between daily Twitch viewership and the overall CS2 Market Index.

* **Null Hypothesis ($H_0$):** There is NO linear correlation between Twitch viewership and the Market Index ($r = 0$).
* **Alternative Hypothesis ($H_1$):** There IS a significant linear correlation ($r \neq 0$).
* **Significance Level ($\alpha$):** 0.05

In [2]:
print("--- TEST 1: PEARSON CORRELATION ---")
corr, p_value = stats.pearsonr(df['ort_izleyici'], df['Market_Index'])

print(f"Correlation Coefficient (r): {corr:.4f}")
print(f"P-Value: {p_value:.4e}")

if p_value < 0.05:
    print("👉 Conclusion: We REJECT the Null Hypothesis. There is a significant linear relationship between Twitch viewership and CS2 item prices.")
else:
    print("👉 Conclusion: We FAIL TO REJECT the Null Hypothesis. No significant linear relationship found.")

--- TEST 1: PEARSON CORRELATION ---
Correlation Coefficient (r): 0.0717
P-Value: 1.3168e-02
👉 Conclusion: We REJECT the Null Hypothesis. There is a significant linear relationship between Twitch viewership and CS2 item prices.


## Test 2: Independent Two-Sample T-Test (Market Value)
**Objective:** To investigate whether the average Market Index during tournament days is significantly different from normal days.

* **Null Hypothesis ($H_0$):** The mean Market Index during Tournaments equals the mean during Normal Periods ($\mu_{tourney} = \mu_{normal}$).
* **Alternative Hypothesis ($H_1$):** The means are significantly different ($\mu_{tourney} \neq \mu_{normal}$).
* **Significance Level ($\alpha$):** 0.05

In [3]:
print("\n--- TEST 2: INDEPENDENT T-TEST ---")
tourney_prices = df[df['Tourney_Type'] != 'Normal Period']['Market_Index']
normal_prices = df[df['Tourney_Type'] == 'Normal Period']['Market_Index']

t_stat, p_value = stats.ttest_ind(tourney_prices, normal_prices, equal_var=False)

print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.4e}")

if p_value < 0.05:
    print("👉 Conclusion: We REJECT the Null Hypothesis. The Market Index during tournaments is statistically different from normal periods.")
else:
    print("👉 Conclusion: We FAIL TO REJECT the Null Hypothesis.")


--- TEST 2: INDEPENDENT T-TEST ---
T-Statistic: 7.3608
P-Value: 3.9522e-13
👉 Conclusion: We REJECT the Null Hypothesis. The Market Index during tournaments is statistically different from normal periods.


## Test 3: One-Way ANOVA (Analysis of Variance)
**Objective:** Since we have three distinct periods (Major Tournaments, S-Tier Tournaments, and Normal Periods), we use ANOVA to see if the market behaves differently across these specific categories.

* **Null Hypothesis ($H_0$):** The mean Market Index is identical across Majors, S-Tiers, and Normal Periods ($\mu_{major} = \mu_{stier} = \mu_{normal}$).
* **Alternative Hypothesis ($H_1$):** At least one of the group means is different.
* **Significance Level ($\alpha$):** 0.05

In [4]:
print("\n--- TEST 3: ONE-WAY ANOVA ---")
major_prices = df[df['Tourney_Type'] == 'Major']['Market_Index']
stier_prices = df[df['Tourney_Type'] == 'S-Tier']['Market_Index']

f_stat, p_value = stats.f_oneway(major_prices, stier_prices, normal_prices)

print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_value:.4e}")

if p_value < 0.05:
    print("👉 Conclusion: We REJECT the Null Hypothesis. The type of tournament (Major vs S-Tier) significantly impacts market prices differently.")
else:
    print("👉 Conclusion: We FAIL TO REJECT the Null Hypothesis.")


--- TEST 3: ONE-WAY ANOVA ---
F-Statistic: 30.3330
P-Value: 1.4144e-13
👉 Conclusion: We REJECT the Null Hypothesis. The type of tournament (Major vs S-Tier) significantly impacts market prices differently.
